In [ ]:
from hcc_release_paths import project_path as _hcc_path
# ============================================================
# Step15A-1
# TCGA-LIHC Step15 Feature Definition Audit
# 建立审计目录 + 盘点现有 Step15 文件
# ============================================================

from pathlib import Path
import pandas as pd
import os

# ------------------------------------------------------------
# 1. Step15 根目录
# ------------------------------------------------------------
STEP15_DIR = Path(_hcc_path('LIHC_RNA_Step15_Machine_Learning'))

# ------------------------------------------------------------
# 2. 创建审计目录
# ------------------------------------------------------------
AUDIT_DIR = STEP15_DIR / "Audit"
RESULTS_DIR = STEP15_DIR / "Results"
INPUT_DIR = STEP15_DIR / "Input"
FIGURES_DIR = STEP15_DIR / "Figures"
MODELS_DIR = STEP15_DIR / "Models"

for d in [
    AUDIT_DIR,
    RESULTS_DIR,
    INPUT_DIR,
    FIGURES_DIR,
    MODELS_DIR
]:
    d.mkdir(parents=True, exist_ok=True)

print("Step15 directory:")
print(STEP15_DIR)
print()

print("Audit directory:")
print(AUDIT_DIR)
print()

# ------------------------------------------------------------
# 3. 盘点所有 CSV / Excel / TXT / MD 文件
# ------------------------------------------------------------
records = []

for path in STEP15_DIR.rglob("*"):
    if path.is_file():
        suffix = path.suffix.lower()

        if suffix in [".csv", ".xlsx", ".xls", ".tsv", ".txt", ".md"]:
            records.append({
                "Relative_Path": str(path.relative_to(STEP15_DIR)),
                "File_Name": path.name,
                "Suffix": suffix,
                "Size_KB": round(path.stat().st_size / 1024, 2)
            })

inventory_df = pd.DataFrame(records)

if len(inventory_df) > 0:
    inventory_df = inventory_df.sort_values(
        ["Suffix", "Relative_Path"]
    ).reset_index(drop=True)

print(f"发现文件数量：{len(inventory_df)}")
print()

display(inventory_df)

# ------------------------------------------------------------
# 4. 保存文件清单
# ------------------------------------------------------------
inventory_path = AUDIT_DIR / "STEP15_existing_files_inventory.csv"

inventory_df.to_csv(
    inventory_path,
    index=False,
    encoding="utf-8-sig"
)

print()
print("文件清单已保存：")
print(inventory_path)

In [ ]:
from hcc_release_paths import project_path as _hcc_path
# ============================================================
# TCGA-LIHC Step15A
# Feature Definition Audit + Version Lock
#
# 目的：
# 1. 核对 Original_Top10
# 2. 核对 Hub18
# 3. 核对 StableGenes10
# 4. 核对 StableBio7
# 5. 建立 Genes10_Bio7
# 6. 建立 Feature Definition Master Table
# 7. 建立 Feature Selection Lineage
# 8. 建立 MASTER_ANALYSIS_TABLE
# 9. 建立 Model Version Log
# 10. 生成 Step15A 审计 Markdown
#
# 注意：
# 本代码不会修改任何原始分析结果。
# ============================================================

from pathlib import Path
from datetime import datetime
import pandas as pd
import numpy as np
import json
import re
import os

# ============================================================
# 0. 路径
# ============================================================

STEP15_DIR = Path(_hcc_path('LIHC_RNA_Step15_Machine_Learning'))

INPUT_DIR = STEP15_DIR / "Input"
RESULTS_DIR = STEP15_DIR / "Results"
FIGURES_DIR = STEP15_DIR / "Figures"
MODELS_DIR = STEP15_DIR / "Models"
AUDIT_DIR = STEP15_DIR / "Audit"

for d in [
    INPUT_DIR,
    RESULTS_DIR,
    FIGURES_DIR,
    MODELS_DIR,
    AUDIT_DIR
]:
    d.mkdir(parents=True, exist_ok=True)

# ============================================================
# 1. 当前版本
# ============================================================

AUDIT_DATE = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

PROJECT_VERSION = "TCGA-LIHC-Step15A-v1.0"

print("=" * 80)
print("TCGA-LIHC Step15A Feature Definition Audit")
print("=" * 80)
print(f"Step15 directory : {STEP15_DIR}")
print(f"Audit directory  : {AUDIT_DIR}")
print(f"Audit date       : {AUDIT_DATE}")
print(f"Project version  : {PROJECT_VERSION}")
print()


# ============================================================
# 2. 通用读取函数
# ============================================================

def read_csv_safe(path):
    path = Path(path)

    if not path.exists():
        print(f"[WARNING] 文件不存在：{path}")
        return None

    try:
        df = pd.read_csv(path)
        print(f"[OK] {path.name:<55} shape={df.shape}")
        return df
    except Exception as e:
        print(f"[ERROR] 无法读取 {path}")
        print(e)
        return None


def find_column(df, candidates):
    """
    在DataFrame中寻找最可能的目标列。
    """
    if df is None or df.empty:
        return None

    normalized = {
        re.sub(r"[^a-z0-9]", "", str(c).lower()): c
        for c in df.columns
    }

    for candidate in candidates:
        key = re.sub(r"[^a-z0-9]", "", str(candidate).lower())

        if key in normalized:
            return normalized[key]

    # 模糊匹配
    for candidate in candidates:
        key = re.sub(r"[^a-z0-9]", "", str(candidate).lower())

        for norm_col, original_col in normalized.items():
            if key in norm_col or norm_col in key:
                return original_col

    return None


def extract_feature_list(df, candidates):
    """
    从结果表中自动识别Feature/Gene列并提取唯一名称。
    """
    col = find_column(df, candidates)

    if col is None:
        return [], None

    values = (
        df[col]
        .dropna()
        .astype(str)
        .str.strip()
    )

    values = values[values != ""]
    values = values.unique().tolist()

    return values, col


# ============================================================
# 3. 历史文件读取
# ============================================================

master_feature_path = (
    INPUT_DIR / "LIHC_Multilevel_Master_Feature_Table.csv"
)

feature_family_path = (
    INPUT_DIR / "Step15_Feature_Family_Summary.csv"
)

split_path = (
    INPUT_DIR / "LIHC_Survival_Train_Test_Split_70_30.csv"
)

gene_uni_path = (
    RESULTS_DIR / "Gene_Training_Univariate_Cox.csv"
)

gene_enet_coef_path = (
    RESULTS_DIR / "Gene_ElasticNet_Coefficients.csv"
)

gene_enet_cv_path = (
    RESULTS_DIR / "Gene_ElasticNet_CV.csv"
)

gene_rsf_path = (
    RESULTS_DIR / "Gene_RSF_Permutation_Importance.csv"
)

gene_integrated_path = (
    RESULTS_DIR / "Gene_MultiMethod_Selection_Integrated.csv"
)

gene_corr_path = (
    RESULTS_DIR / "Gene_Training_High_Correlation_Pairs.csv"
)

stable_gene_path = (
    RESULTS_DIR / "Stable_Prognostic_Genes_10.csv"
)

bio_uni_path = (
    RESULTS_DIR / "Biological_Features_Training_Univariate_Cox.csv"
)

bio_enet_coef_path = (
    RESULTS_DIR / "Biological_ElasticNet_Coefficients.csv"
)

bio_enet_cv_path = (
    RESULTS_DIR / "Biological_ElasticNet_CV.csv"
)

bio_rsf_path = (
    RESULTS_DIR / "Biological_RSF_Permutation_Importance.csv"
)

bio_integrated_path = (
    RESULTS_DIR / "Biological_MultiMethod_Selection_Integrated.csv"
)

bio_corr_path = (
    RESULTS_DIR / "Biological_Features_High_Correlation_Pairs.csv"
)

stable_bio_path = (
    RESULTS_DIR / "Stable_Biological_Features.csv"
)

integrated_benchmark_path = (
    RESULTS_DIR / "Integrated_Feature_Sets_CoxPH_Benchmark.csv"
)

core_benchmark_path = (
    RESULTS_DIR / "Core_Feature_Sets_CoxPH_Benchmark.csv"
)

multi_algorithm_path = (
    RESULTS_DIR / "MultiAlgorithm_Survival_Model_Comparison.csv"
)

td_auc_path = (
    RESULTS_DIR / "Genes10_Bio7_TimeDependent_AUC.csv"
)

km_path = (
    RESULTS_DIR / "Final3Models_Test_KM_Results.csv"
)

calibration_path = (
    RESULTS_DIR / "Final3Models_Calibration_Data.csv"
)

dca_path = (
    RESULTS_DIR / "Final3Models_DCA_Data.csv"
)

# ------------------------------------------------------------
# 实际读取
# ------------------------------------------------------------

master_feature_df = read_csv_safe(master_feature_path)
feature_family_df = read_csv_safe(feature_family_path)
split_df = read_csv_safe(split_path)

gene_uni_df = read_csv_safe(gene_uni_path)
gene_enet_coef_df = read_csv_safe(gene_enet_coef_path)
gene_enet_cv_df = read_csv_safe(gene_enet_cv_path)
gene_rsf_df = read_csv_safe(gene_rsf_path)
gene_integrated_df = read_csv_safe(gene_integrated_path)
gene_corr_df = read_csv_safe(gene_corr_path)
stable_gene_df = read_csv_safe(stable_gene_path)

bio_uni_df = read_csv_safe(bio_uni_path)
bio_enet_coef_df = read_csv_safe(bio_enet_coef_path)
bio_enet_cv_df = read_csv_safe(bio_enet_cv_path)
bio_rsf_df = read_csv_safe(bio_rsf_path)
bio_integrated_df = read_csv_safe(bio_integrated_path)
bio_corr_df = read_csv_safe(bio_corr_path)
stable_bio_df = read_csv_safe(stable_bio_path)

integrated_benchmark_df = read_csv_safe(integrated_benchmark_path)
core_benchmark_df = read_csv_safe(core_benchmark_path)
multi_algorithm_df = read_csv_safe(multi_algorithm_path)
td_auc_df = read_csv_safe(td_auc_path)
km_df = read_csv_safe(km_path)
calibration_df = read_csv_safe(calibration_path)
dca_df = read_csv_safe(dca_path)

print()


# ============================================================
# 4. LOCKED DEFINITIONS
# ============================================================

# ------------------------------------------------------------
# 4.1 Original Final Top10
#
# 历史来源：
# TCGA-LIHC早期OS预后模型
# 20622 genes
# -> 3761 univariate Cox candidates
# -> LASSO-CV
# -> 79 genes
# -> |LASSO coefficient| Top10
# -> multivariable CoxPH
# -> Original RiskScore
# ------------------------------------------------------------

ORIGINAL_TOP10 = [
    "HOXD9",
    "RFESD",
    "RPSAP53",
    "TRGC1",
    "FAM227B",
    "AC015871.6",
    "LINC00324",
    "ACOXL",
    "PTMAP4",
    "ANKRD20A7P"
]

# ------------------------------------------------------------
# 4.2 Locked Hub18
#
# 严格模块归属：
# |MM| >= 0.80
# |GS| >= 0.20
# assigned module only
# Hub_score = |MM| × |GS|
# 每个关键模块取Top6
# ------------------------------------------------------------

HUB18_BY_MODULE = {
    "Blue": [
        "CENPA",
        "KIF2C",
        "CCNB1",
        "CDC20",
        "PRR11",
        "CDCA8"
    ],
    "Red": [
        "SLC52A2",
        "RPL8",
        "BOP1",
        "ZFAS1",
        "SNHG6",
        "RPL7P9"
    ],
    "Brown": [
        "CYP8B1",
        "GLYATL1",
        "PLG",
        "ACSM2A",
        "ABAT",
        "MAT1A"
    ]
}

HUB18 = [
    gene
    for genes in HUB18_BY_MODULE.values()
    for gene in genes
]

# ------------------------------------------------------------
# 4.3 StableGenes10
#
# 28-gene pool:
# Original Top10 + Hub18
#
# Train-only:
# Univariate Cox
# ElasticNet Cox
# RSF permutation importance
#
# Selection_Count = 3
# ------------------------------------------------------------

LOCKED_STABLE_GENES10 = [
    "HOXD9",
    "RPSAP53",
    "RFESD",
    "LINC00324",
    "TRGC1",
    "ANKRD20A7P",
    "FAM227B",
    "BOP1",
    "GLYATL1",
    "PRR11"
]

# ------------------------------------------------------------
# 4.4 StableBio7
#
# Biological feature pool:
# WGCNA3
# + Subtype3
# + Immune30
# + Hallmark GSVA ~50
#
# Train-only:
# Univariate Cox
# ElasticNet Cox
# RSF
#
# Selection_Count = 3
# ------------------------------------------------------------

LOCKED_STABLE_BIO7 = [
    "mTORC1 Signaling",
    "Checkpoint_TNFRSF4",
    "Cell_B_cells",
    "Function_HLA_activity",
    "Checkpoint_CD276",
    "Pancreas Beta Cells",
    "Spermatogenesis"
]

# 类型
STABLE_BIO7_TYPE = {
    "mTORC1 Signaling": "Hallmark_GSVA",
    "Checkpoint_TNFRSF4": "Immune",
    "Cell_B_cells": "Immune",
    "Function_HLA_activity": "Immune",
    "Checkpoint_CD276": "Immune",
    "Pancreas Beta Cells": "Hallmark_GSVA",
    "Spermatogenesis": "Hallmark_GSVA"
}

# ============================================================
# 5. Genes10_Bio7
# ============================================================

GENES10_BIO7 = LOCKED_STABLE_GENES10 + LOCKED_STABLE_BIO7

assert len(LOCKED_STABLE_GENES10) == 10
assert len(LOCKED_STABLE_BIO7) == 7
assert len(GENES10_BIO7) == 17
assert len(set(GENES10_BIO7)) == 17

print("=" * 80)
print("LOCKED FEATURE DEFINITIONS")
print("=" * 80)

print(f"Original Top10 : {len(ORIGINAL_TOP10)}")
print(f"Hub18          : {len(HUB18)}")
print(f"StableGenes10  : {len(LOCKED_STABLE_GENES10)}")
print(f"StableBio7     : {len(LOCKED_STABLE_BIO7)}")
print(f"Genes10_Bio7   : {len(GENES10_BIO7)}")

print()
print("Genes10_Bio7 是否存在重复：",
      len(GENES10_BIO7) != len(set(GENES10_BIO7)))

print()


# ============================================================
# 6. 自动读取 StableGenes10 历史文件并核对
# ============================================================

actual_stable_genes, stable_gene_col = extract_feature_list(
    stable_gene_df,
    ["Gene", "Genes", "Feature", "Gene_Name"]
)

print("=" * 80)
print("StableGenes10 HISTORY CHECK")
print("=" * 80)

print("历史文件识别列：", stable_gene_col)
print("历史文件内容：", actual_stable_genes)
print("锁定定义：", LOCKED_STABLE_GENES10)

if set(actual_stable_genes) == set(LOCKED_STABLE_GENES10):
    stable_gene_status = "PASS"
    print("PASS：StableGenes10 与历史文件完全一致。")
else:
    stable_gene_status = "WARNING"
    print("WARNING：StableGenes10 与历史文件存在差异。")
    print("历史文件独有：",
          sorted(set(actual_stable_genes) - set(LOCKED_STABLE_GENES10)))
    print("锁定定义独有：",
          sorted(set(LOCKED_STABLE_GENES10) - set(actual_stable_genes)))

print()


# ============================================================
# 7. 自动读取 StableBio7 历史文件并核对
# ============================================================

actual_stable_bio, stable_bio_col = extract_feature_list(
    stable_bio_df,
    ["Feature", "Biological_Feature", "Feature_Name", "Gene"]
)

print("=" * 80)
print("StableBio7 HISTORY CHECK")
print("=" * 80)

print("历史文件识别列：", stable_bio_col)
print("历史文件内容：", actual_stable_bio)
print("锁定定义：", LOCKED_STABLE_BIO7)

if set(actual_stable_bio) == set(LOCKED_STABLE_BIO7):
    stable_bio_status = "PASS"
    print("PASS：StableBio7 与历史文件完全一致。")
else:
    stable_bio_status = "WARNING"
    print("WARNING：StableBio7 与历史文件存在差异。")
    print("历史文件独有：",
          sorted(set(actual_stable_bio) - set(LOCKED_STABLE_BIO7)))
    print("锁定定义独有：",
          sorted(set(LOCKED_STABLE_BIO7) - set(actual_stable_bio)))

print()


# ============================================================
# 8. 构建 Gene Definition Master Table
# ============================================================

gene_rows = []

# ------------------------------------------------------------
# Original Top10
# ------------------------------------------------------------

for i, gene in enumerate(ORIGINAL_TOP10, start=1):

    gene_rows.append({
        "Gene": gene,
        "Gene_Set": "Original_Top10",
        "Role": "Original prognostic signature",
        "Source_Step": "Early survival model / Step8",
        "Source_File": (
            r"E:\TCGA\LIHC_RNA_Step8_Final_Model\Results"
            r"\Final_Top10_RiskScore.csv"
        ),
        "Candidate_Pool": "TCGA-LIHC 20622 genes",
        "Selection_Method": (
            "Univariate Cox -> LASSO-CV -> "
            "|LASSO coefficient| Top10 -> multivariable CoxPH"
        ),
        "Training_Only": "No",
        "Selection_Count": np.nan,
        "Module": np.nan,
        "Final_Status": "LOCKED",
        "Downstream_Role": "Original RiskScore"
    })

# ------------------------------------------------------------
# Hub18
# ------------------------------------------------------------

for module, genes in HUB18_BY_MODULE.items():

    for i, gene in enumerate(genes, start=1):

        gene_rows.append({
            "Gene": gene,
            "Gene_Set": "Hub18",
            "Role": "WGCNA module hub candidate",
            "Source_Step": "Step12 WGCNA / Step13 input",
            "Source_File": (
                r"E:\TCGA\LIHC_RNA_Step13_Molecular_Subtyping"
                r"\Input\LIHC_Strict_Assigned_Hub_Table.csv"
            ),
            "Candidate_Pool": "Top5000 MAD expression genes",
            "Selection_Method": (
                "|MM| >= 0.80 + |GS| >= 0.20; "
                "assigned module only; "
                "rank by |MM| x |GS|; Top6 per key module"
            ),
            "Training_Only": "No",
            "Selection_Count": np.nan,
            "Module": module,
            "Final_Status": "LOCKED",
            "Downstream_Role": "Hub gene / Step15 28-gene pool"
        })

# ------------------------------------------------------------
# StableGenes10
# ------------------------------------------------------------

for gene in LOCKED_STABLE_GENES10:

    if gene in ORIGINAL_TOP10:
        origin = "Original_Top10"
    elif gene in HUB18:
        origin = "Hub18"
    else:
        origin = "Unexpected"

    gene_rows.append({
        "Gene": gene,
        "Gene_Set": "StableGenes10",
        "Role": "ML-derived stable prognostic gene",
        "Source_Step": "Step15 Machine Learning",
        "Source_File": (
            r"E:\TCGA\LIHC_RNA_Step15_Machine_Learning"
            r"\Results\Stable_Prognostic_Genes_10.csv"
        ),
        "Candidate_Pool": "Original_Top10 + Hub18 = 28 genes",
        "Selection_Method": (
            "Train-only Univariate Cox + ElasticNet Cox + "
            "RSF permutation importance; Selection_Count = 3"
        ),
        "Training_Only": "Yes",
        "Selection_Count": 3,
        "Module": np.nan,
        "Final_Status": "LOCKED",
        "Downstream_Role": (
            f"StableGenes10; original source = {origin}"
        )
    })

gene_definition_df = pd.DataFrame(gene_rows)

gene_definition_path = (
    RESULTS_DIR / "Gene_Definition_Master_Table.csv"
)

gene_definition_df.to_csv(
    gene_definition_path,
    index=False,
    encoding="utf-8-sig"
)

print("=" * 80)
print("Gene Definition Master Table")
print("=" * 80)
print(gene_definition_df.shape)
print(gene_definition_path)
print()


# ============================================================
# 9. Feature Definition Master Table
# ============================================================

feature_rows = []

# ------------------------------------------------------------
# StableGenes10
# ------------------------------------------------------------

for gene in LOCKED_STABLE_GENES10:

    feature_rows.append({
        "Feature": gene,
        "Feature_Type": "Gene",
        "Feature_Set": "StableGenes10",
        "Source_Step": "Step15",
        "Source_Module": np.nan,
        "Selection_Method": (
            "Univariate Cox + ElasticNet Cox + RSF"
        ),
        "Selection_Count": 3,
        "Training_Only": "Yes",
        "Role": "ML-derived stable gene",
        "Status": "LOCKED"
    })

# ------------------------------------------------------------
# StableBio7
# ------------------------------------------------------------

for feature in LOCKED_STABLE_BIO7:

    feature_rows.append({
        "Feature": feature,
        "Feature_Type": STABLE_BIO7_TYPE[feature],
        "Feature_Set": "StableBio7",
        "Source_Step": (
            "Step11 Immune"
            if STABLE_BIO7_TYPE[feature] == "Immune"
            else "Step14 Pathway"
        ),
        "Source_Module": np.nan,
        "Selection_Method": (
            "Univariate Cox + ElasticNet Cox + RSF"
        ),
        "Selection_Count": 3,
        "Training_Only": "Yes",
        "Role": "ML-derived stable biological feature",
        "Status": "LOCKED"
    })

# ------------------------------------------------------------
# Genes10_Bio7
# ------------------------------------------------------------

for feature in GENES10_BIO7:

    if feature in LOCKED_STABLE_GENES10:
        feature_type = "Gene"
        source_set = "StableGenes10"
    else:
        feature_type = STABLE_BIO7_TYPE[feature]
        source_set = "StableBio7"

    feature_rows.append({
        "Feature": feature,
        "Feature_Type": feature_type,
        "Feature_Set": "Genes10_Bio7",
        "Source_Step": "Step15 integrated model",
        "Source_Module": source_set,
        "Selection_Method": (
            "Union of StableGenes10 + StableBio7"
        ),
        "Selection_Count": np.nan,
        "Training_Only": "Yes",
        "Role": "Integrated model feature",
        "Status": "LOCKED"
    })

feature_definition_df = pd.DataFrame(feature_rows)

feature_definition_path = (
    RESULTS_DIR / "Feature_Definition_Master_Table.csv"
)

feature_definition_df.to_csv(
    feature_definition_path,
    index=False,
    encoding="utf-8-sig"
)

print("=" * 80)
print("Feature Definition Master Table")
print("=" * 80)
print(feature_definition_df.shape)
print(feature_definition_path)
print()


# ============================================================
# 10. Feature Selection Lineage
# ============================================================

lineage_rows = [

    {
        "Object": "Original_Top10",
        "Object_Type": "Gene_Set",
        "N_Features": 10,
        "Source_Step": "Early survival model / Step8",
        "Candidate_Pool": "TCGA-LIHC 20622 genes",
        "Selection_Process": (
            "Univariate Cox -> 3761 candidates -> "
            "LASSO-CV -> 79 genes -> "
            "|LASSO coefficient| Top10 -> multivariable CoxPH"
        ),
        "Training_N": np.nan,
        "Training_Only": "No",
        "Final_Output": "Original RiskScore",
        "Source_File": (
            r"E:\TCGA\LIHC_RNA_Step8_Final_Model\Results"
            r"\Final_Top10_RiskScore.csv"
        ),
        "Status": "LOCKED"
    },

    {
        "Object": "Hub18",
        "Object_Type": "Gene_Set",
        "N_Features": 18,
        "Source_Step": "Step12 WGCNA / Step13",
        "Candidate_Pool": "Top5000 MAD genes",
        "Selection_Process": (
            "|MM| >= 0.80 + |GS| >= 0.20; "
            "assigned module only; "
            "Hub_score = |MM| x |GS|; "
            "Top6 per Blue/Red/Brown"
        ),
        "Training_N": np.nan,
        "Training_Only": "No",
        "Final_Output": "Hub18",
        "Source_File": (
            r"E:\TCGA\LIHC_RNA_Step13_Molecular_Subtyping"
            r"\Input\LIHC_Strict_Assigned_Hub_Table.csv"
        ),
        "Status": "LOCKED"
    },

    {
        "Object": "StableGenes10",
        "Object_Type": "Gene_Set",
        "N_Features": 10,
        "Source_Step": "Step15 Machine Learning",
        "Candidate_Pool": "Original_Top10 + Hub18 = 28 genes",
        "Selection_Process": (
            "Train-only Univariate Cox + "
            "ElasticNet Cox + RSF; "
            "Selection_Count = 3"
        ),
        "Training_N": 238,
        "Training_Only": "Yes",
        "Final_Output": "StableGenes10",
        "Source_File": (
            r"E:\TCGA\LIHC_RNA_Step15_Machine_Learning"
            r"\Results\Stable_Prognostic_Genes_10.csv"
        ),
        "Status": stable_gene_status
    },

    {
        "Object": "StableBio7",
        "Object_Type": "Biological_Feature_Set",
        "N_Features": 7,
        "Source_Step": "Step15 Machine Learning",
        "Candidate_Pool": (
            "WGCNA3 + Subtype3 + Immune30 + "
            "Hallmark GSVA ~50"
        ),
        "Selection_Process": (
            "Train-only Univariate Cox P<0.05 -> "
            "24 candidates -> ElasticNet Cox + RSF -> "
            "Selection_Count = 3"
        ),
        "Training_N": 238,
        "Training_Only": "Yes",
        "Final_Output": "StableBio7",
        "Source_File": (
            r"E:\TCGA\LIHC_RNA_Step15_Machine_Learning"
            r"\Results\Stable_Biological_Features.csv"
        ),
        "Status": stable_bio_status
    },

    {
        "Object": "Genes10_Bio7",
        "Object_Type": "Integrated_Feature_Set",
        "N_Features": 17,
        "Source_Step": "Step15 Machine Learning",
        "Candidate_Pool": (
            "StableGenes10 + StableBio7"
        ),
        "Selection_Process": (
            "Union of 10 stable genes and "
            "7 stable biological features"
        ),
        "Training_N": 238,
        "Training_Only": "Yes",
        "Final_Output": "17-feature integrated model",
        "Source_File": (
            r"E:\TCGA\LIHC_RNA_Step15_Machine_Learning"
            r"\Results\Integrated_Feature_Sets_CoxPH_Benchmark.csv"
        ),
        "Status": "LOCKED"
    }
]

lineage_df = pd.DataFrame(lineage_rows)

lineage_path = (
    RESULTS_DIR / "FEATURE_SELECTION_LINEAGE.csv"
)

lineage_df.to_csv(
    lineage_path,
    index=False,
    encoding="utf-8-sig"
)

print("=" * 80)
print("Feature Selection Lineage")
print("=" * 80)
display(lineage_df)
print()
print(lineage_path)
print()


# ============================================================
# 11. 建立 MASTER_ANALYSIS_TABLE
# ============================================================

master_rows = []

# ------------------------------------------------------------
# Feature objects
# ------------------------------------------------------------

master_rows.extend([
    {
        "Record_Type": "Feature_Definition",
        "Object": "Original_Top10",
        "N_Features": 10,
        "Feature_Type": "Gene",
        "Role": "Original RiskScore",
        "Source_File": str(
            r"E:\TCGA\LIHC_RNA_Step8_Final_Model\Results"
            r"\Final_Top10_RiskScore.csv"
        ),
        "Status": "LOCKED"
    },
    {
        "Record_Type": "Feature_Definition",
        "Object": "Hub18",
        "N_Features": 18,
        "Feature_Type": "Gene",
        "Role": "WGCNA hub candidates",
        "Source_File": str(
            r"E:\TCGA\LIHC_RNA_Step13_Molecular_Subtyping"
            r"\Input\LIHC_Strict_Assigned_Hub_Table.csv"
        ),
        "Status": "LOCKED"
    },
    {
        "Record_Type": "Feature_Definition",
        "Object": "StableGenes10",
        "N_Features": 10,
        "Feature_Type": "Gene",
        "Role": "ML-derived stable genes",
        "Source_File": str(stable_gene_path),
        "Status": stable_gene_status
    },
    {
        "Record_Type": "Feature_Definition",
        "Object": "StableBio7",
        "N_Features": 7,
        "Feature_Type": "Biological Feature",
        "Role": "ML-derived stable biological features",
        "Source_File": str(stable_bio_path),
        "Status": stable_bio_status
    },
    {
        "Record_Type": "Feature_Definition",
        "Object": "Genes10_Bio7",
        "N_Features": 17,
        "Feature_Type": "Integrated Feature Set",
        "Role": "Integrated survival model",
        "Source_File": str(integrated_benchmark_path),
        "Status": "LOCKED"
    }
])

# ------------------------------------------------------------
# 历史结果文件
# ------------------------------------------------------------

analysis_files = [
    ("Gene_Univariate_Cox", gene_uni_path),
    ("Gene_ElasticNet_CV", gene_enet_cv_path),
    ("Gene_ElasticNet_Coefficients", gene_enet_coef_path),
    ("Gene_RSF_Importance", gene_rsf_path),
    ("Gene_MultiMethod_Selection", gene_integrated_path),
    ("Gene_Correlation", gene_corr_path),

    ("Bio_Univariate_Cox", bio_uni_path),
    ("Bio_ElasticNet_CV", bio_enet_cv_path),
    ("Bio_ElasticNet_Coefficients", bio_enet_coef_path),
    ("Bio_RSF_Importance", bio_rsf_path),
    ("Bio_MultiMethod_Selection", bio_integrated_path),
    ("Bio_Correlation", bio_corr_path),

    ("Core_Feature_CoxPH_Benchmark", core_benchmark_path),
    ("Integrated_Feature_CoxPH_Benchmark", integrated_benchmark_path),
    ("MultiAlgorithm_Model_Comparison", multi_algorithm_path),
    ("Genes10_Bio7_TimeDependent_AUC", td_auc_path),
    ("Final3Models_Test_KM", km_path),
    ("Final3Models_Calibration", calibration_path),
    ("Final3Models_DCA", dca_path)
]

for analysis_name, path in analysis_files:

    master_rows.append({
        "Record_Type": "Historical_Result",
        "Object": analysis_name,
        "N_Features": np.nan,
        "Feature_Type": "Analysis_Result",
        "Role": "Previously completed Step15 result",
        "Source_File": str(path),
        "Status": (
            "EXISTS"
            if path.exists()
            else "MISSING"
        )
    })

master_analysis_df = pd.DataFrame(master_rows)

master_analysis_path = (
    RESULTS_DIR / "MASTER_ANALYSIS_TABLE.csv"
)

master_analysis_df.to_csv(
    master_analysis_path,
    index=False,
    encoding="utf-8-sig"
)

print("=" * 80)
print("MASTER_ANALYSIS_TABLE")
print("=" * 80)
print(master_analysis_df.shape)
display(master_analysis_df)
print()
print(master_analysis_path)
print()


# ============================================================
# 12. MODEL VERSION LOG
# ============================================================

version_rows = [

    {
        "Version": "Original_RiskScore_v1.0",
        "Object": "Original_Top10",
        "Definition": "Original 10-gene prognostic signature",
        "Purpose": "Historical benchmark / original RiskScore",
        "Can_Modify": "NO",
        "External_Validation": "Pending",
        "Status": "LOCKED"
    },

    {
        "Version": "Hub18_v1.0",
        "Object": "Hub18",
        "Definition": "Strict assigned-module WGCNA hubs",
        "Purpose": "Candidate gene pool for Step15",
        "Can_Modify": "NO",
        "External_Validation": "Not yet",
        "Status": "LOCKED"
    },

    {
        "Version": "StableGenes10_v1.0",
        "Object": "StableGenes10",
        "Definition": (
            "10 genes selected by "
            "Univariate Cox + ElasticNet + RSF"
        ),
        "Purpose": "ML-derived stable gene signature",
        "Can_Modify": "NO",
        "External_Validation": "Pending",
        "Status": stable_gene_status
    },

    {
        "Version": "StableBio7_v1.0",
        "Object": "StableBio7",
        "Definition": (
            "7 biological features selected by "
            "Univariate Cox + ElasticNet + RSF"
        ),
        "Purpose": "Stable biological feature set",
        "Can_Modify": "NO",
        "External_Validation": "Pending",
        "Status": stable_bio_status
    },

    {
        "Version": "Genes10_Bio7_v1.0",
        "Object": "Genes10_Bio7",
        "Definition": "10 stable genes + 7 stable biological features",
        "Purpose": "Integrated survival model",
        "Can_Modify": "NO",
        "External_Validation": "Pending",
        "Status": "LOCKED"
    },

    {
        "Version": "Step15_Stability_v2.0",
        "Object": "Future ML stability analysis",
        "Definition": (
            "Repeated CV + bootstrap + "
            "train-only feature selection"
        ),
        "Purpose": "Confirm selection and model stability",
        "Can_Modify": "YES",
        "External_Validation": "Not applicable yet",
        "Status": "NEXT"
    },

    {
        "Version": "Core_Driver_Genes",
        "Object": "Future 1-3 genes",
        "Definition": (
            "Genes surviving external validation, "
            "scRNA localization, GRN and spatial validation"
        ),
        "Purpose": "Mechanistic candidates",
        "Can_Modify": "YES",
        "External_Validation": "Required",
        "Status": "NOT_DEFINED"
    }
]

version_df = pd.DataFrame(version_rows)

version_path = (
    RESULTS_DIR / "MODEL_VERSION_LOG.csv"
)

version_df.to_csv(
    version_path,
    index=False,
    encoding="utf-8-sig"
)

print("=" * 80)
print("MODEL_VERSION_LOG")
print("=" * 80)
display(version_df)
print()
print(version_path)
print()


# ============================================================
# 13. 生成完整 STEP15A 审计 Markdown
# ============================================================

audit_md = f"""
# TCGA-LIHC Step15A Feature Definition Audit

**Project version:** {PROJECT_VERSION}

**Audit date:** {AUDIT_DATE}

---

## 1. Purpose

This audit locks the definitions and lineage of all major feature sets
developed during TCGA-LIHC Step15.

No original analysis result is modified.

---

# 2. Original Top10

## Definition

The Original Top10 originated from the early TCGA-LIHC overall-survival
prognostic modeling workflow:

20622 genes
→ 3761 univariate Cox candidates
→ LASSO-CV
→ 79 genes
→ top 10 genes ranked by absolute LASSO coefficient
→ multivariable CoxPH
→ Original RiskScore.

## Locked genes

{", ".join(ORIGINAL_TOP10)}

## Role

Original prognostic signature / Original RiskScore.

## Status

LOCKED.

This signature must not be silently replaced by StableGenes10.

---

# 3. Hub18

Hub18 originated from the strict WGCNA hub-gene definition.

Criteria:

- assigned module only
- |MM| >= 0.80
- |GS| >= 0.20
- Hub_score = |MM| × |GS|
- top 6 genes from each key module

Key modules:

- Blue
- Red
- Brown

## Blue

{", ".join(HUB18_BY_MODULE["Blue"])}

## Red

{", ".join(HUB18_BY_MODULE["Red"])}

## Brown

{", ".join(HUB18_BY_MODULE["Brown"])}

Total = 18 genes.

---

# 4. StableGenes10

StableGenes10 was derived during Step15 Machine Learning.

Candidate pool:

Original Top10 + Hub18 = 28 genes.

Training cohort:

238 patients.

Selection methods:

1. Training-only univariate Cox
2. Elastic Net Cox
3. Random Survival Forest permutation importance

Final criterion:

Selection_Count = 3.

## Locked StableGenes10

{", ".join(LOCKED_STABLE_GENES10)}

Total = 10 genes.

## Important distinction

StableGenes10 is NOT a replacement of Original Top10.

Original Top10 = historical RiskScore signature.

StableGenes10 = ML-derived stable gene signature.

---

# 5. StableBio7

StableBio7 is NOT a gene set.

It is a biological feature set.

Initial biological feature pool:

- WGCNA module features: 3
- Molecular subtype features: 3
- Immune features: 30
- Hallmark GSVA features: approximately 50

Total approximately 86 biological features.

Training cohort:

238 patients.

Initial univariate Cox screening:

P < 0.05 → 24 candidate biological features.

Then:

Univariate Cox + ElasticNet Cox + RSF.

Final criterion:

Selection_Count = 3.

## Locked StableBio7

{", ".join(LOCKED_STABLE_BIO7)}

### Feature types

"""

for feature in LOCKED_STABLE_BIO7:
    audit_md += (
        f"- **{feature}** — "
        f"{STABLE_BIO7_TYPE[feature]}\\n"
    )

audit_md += f"""

Total = 7 biological features.

Important:

Selection_Count = 3 means that the feature was selected by all three
selection methods.

It does NOT mean that every StableBio7 feature has FDR < 0.05 in the
univariate Cox analysis.

---

# 6. Genes10_Bio7

Genes10_Bio7 is an integrated feature set.

StableGenes10:

10 genes

+

StableBio7:

7 biological features

=

17 unique features.

It is therefore NOT a 17-gene signature.

It should be described as:

**17-feature integrated signature / integrated feature set**

---

# 7. Current research roles

| Object | Role |
|---|---|
| Original_Top10 | Original RiskScore |
| Hub18 | WGCNA hub candidate pool |
| StableGenes10 | ML-derived stable gene signature |
| StableBio7 | ML-derived stable biological feature set |
| Genes10_Bio7 | 17-feature integrated survival model |
| Core_Driver_Genes | Not yet defined |

---

# 8. Important methodological limitation

The current StableGenes10 and StableBio7 definitions were developed
within the TCGA-LIHC discovery cohort.

Therefore:

- they are not independent external validation results;
- they should not yet be described as externally validated signatures;
- repeated CV and bootstrap stability analysis are still required;
- external GEO/ICGC validation remains mandatory.

---

# 9. Next Step

The next analysis phase is:

**Step15B — Repeated Cross-Validation + Bootstrap Stability**

Requirements:

1. Train-only feature selection.
2. Repeated K-fold cross-validation.
3. Feature selection repeated independently inside each training fold.
4. Bootstrap stability analysis.
5. C-index confidence intervals.
6. Time-dependent AUC confidence intervals.
7. Feature selection frequency.
8. Comparison of CoxPH, ElasticNet Cox and RSF.

The current single 70/30 split remains an exploratory development result.

---

# 10. Future research direction

After Step15 is locked:

TCGA Discovery
→ repeated-CV/bootstrapped ML stability
→ GEO validation
→ ICGC validation
→ postoperative RFS / early recurrence
→ scRNA malignant-cell state mapping
→ trajectory / GRN
→ cell-cell communication
→ spatial transcriptomics
→ spatial niche
→ 1–3 core driver genes
→ virtual perturbation
→ IHC / mIF.

The final mechanistic driver genes are intentionally NOT defined at this stage.

---

# 11. Version lock

Original_Top10: LOCKED

Hub18: LOCKED

StableGenes10: {stable_gene_status}

StableBio7: {stable_bio_status}

Genes10_Bio7: LOCKED

Future Core_Driver_Genes: NOT_DEFINED
"""

audit_md_path = (
    AUDIT_DIR / "STEP15A_FEATURE_DEFINITION_AUDIT.md"
)

with open(
    audit_md_path,
    "w",
    encoding="utf-8"
) as f:
    f.write(audit_md)

print("=" * 80)
print("Audit Markdown")
print("=" * 80)
print(audit_md_path)
print()


# ============================================================
# 14. 保存最终锁定的JSON版本
# ============================================================

locked_definition = {
    "project_version": PROJECT_VERSION,
    "audit_date": AUDIT_DATE,

    "Original_Top10": ORIGINAL_TOP10,

    "Hub18": HUB18_BY_MODULE,

    "StableGenes10": LOCKED_STABLE_GENES10,

    "StableBio7": {
        feature: {
            "type": STABLE_BIO7_TYPE[feature]
        }
        for feature in LOCKED_STABLE_BIO7
    },

    "Genes10_Bio7": GENES10_BIO7,

    "counts": {
        "Original_Top10": len(ORIGINAL_TOP10),
        "Hub18": len(HUB18),
        "StableGenes10": len(LOCKED_STABLE_GENES10),
        "StableBio7": len(LOCKED_STABLE_BIO7),
        "Genes10_Bio7": len(GENES10_BIO7)
    }
}

json_path = (
    AUDIT_DIR / "LOCKED_FEATURE_DEFINITIONS_v1.0.json"
)

with open(
    json_path,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        locked_definition,
        f,
        ensure_ascii=False,
        indent=2
    )

print("=" * 80)
print("LOCKED JSON")
print("=" * 80)
print(json_path)
print()


# ============================================================
# 15. 最终一致性检查
# ============================================================

print("=" * 80)
print("FINAL CONSISTENCY CHECK")
print("=" * 80)

checks = {
    "Original Top10 = 10 genes":
        len(ORIGINAL_TOP10) == 10,

    "Hub18 = 18 genes":
        len(HUB18) == 18,

    "StableGenes10 = 10 genes":
        len(LOCKED_STABLE_GENES10) == 10,

    "StableBio7 = 7 features":
        len(LOCKED_STABLE_BIO7) == 7,

    "Genes10_Bio7 = 17 features":
        len(GENES10_BIO7) == 17,

    "Genes10_Bio7 unique":
        len(set(GENES10_BIO7)) == 17,

    "StableGenes10 subset of 28-gene pool":
        set(LOCKED_STABLE_GENES10).issubset(
            set(ORIGINAL_TOP10 + HUB18)
        ),

    "StableBio7 all biological features":
        all(
            x in STABLE_BIO7_TYPE
            for x in LOCKED_STABLE_BIO7
        )
}

for name, result in checks.items():
    print(f"{'PASS' if result else 'FAIL':<6} | {name}")

print()

# ============================================================
# 16. 输出最终文件
# ============================================================

print("=" * 80)
print("STEP15A COMPLETE")
print("=" * 80)

print()
print("Results:")
print(f"1. {gene_definition_path}")
print(f"2. {feature_definition_path}")
print(f"3. {lineage_path}")
print(f"4. {master_analysis_path}")
print(f"5. {version_path}")

print()
print("Audit:")
print(f"6. {audit_md_path}")
print(f"7. {json_path}")

print()
print("下一阶段：")
print("Step15B = Repeated CV + Bootstrap + Train-only Feature Selection")